In [3]:
import pandas as pd
import numpy as np
import os

In [2]:
import pandas as pd
import numpy as np
import os
 
RAW_DIR = "."            # <-- change this to your data folder if needed
OUT_DIR = "./cleaned"    # <-- cleaned files + report will be saved here
os.makedirs(OUT_DIR, exist_ok=True)
 
report_lines = []
 
def log(msg):
    print(msg)
    report_lines.append(msg)
 
 
def strip_and_standardize_strings(df, cols):
    """Trim whitespace and normalize casing for categorical/text columns."""
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    return df
 
 
def report_and_drop_duplicates(df, name, subset=None):
    before = len(df)
    df = df.drop_duplicates(subset=subset)
    after = len(df)
    if before != after:
        log(f"[{name}] Removed {before - after} duplicate rows"
            f"{f' (based on {subset})' if subset else ''}.")
    return df
 
 
def report_and_drop_nulls(df, name, critical_cols):
    """Drop rows missing values in columns that must never be null."""
    before = len(df)
    existing = [c for c in critical_cols if c in df.columns]
    df = df.dropna(subset=existing)
    after = len(df)
    if before != after:
        log(f"[{name}] Dropped {before - after} rows with nulls in {existing}.")
    return df
 
 
def flag_invalid_range(df, name, col, min_val=None, max_val=None, drop=True):
    """Flag/remove rows where col falls outside a valid range."""
    if col not in df.columns:
        return df
    mask = pd.Series(True, index=df.index)
    if min_val is not None:
        mask &= df[col] >= min_val
    if max_val is not None:
        mask &= df[col] <= max_val
    invalid = (~mask).sum()
    if invalid > 0:
        log(f"[{name}] {invalid} rows had '{col}' outside "
            f"[{min_val}, {max_val}] — {'dropped' if drop else 'flagged only'}.")
        if drop:
            df = df[mask]
    return df
 
 
# ---------------------------------------------------------------------
# 1. CUSTOMERS
# ---------------------------------------------------------------------
log("Processing blinkit_customers.csv ...")
customers = pd.read_csv(f"{RAW_DIR}/blinkit_customers.csv")
 
customers = strip_and_standardize_strings(
    customers, ["customer_name", "email", "address", "area", "customer_segment"]
)
customers["area"] = customers["area"].str.title()
customers["customer_segment"] = customers["customer_segment"].str.title()
customers["registration_date"] = pd.to_datetime(
    customers["registration_date"], errors="coerce"
)
customers = report_and_drop_nulls(
    customers, "customers", ["customer_id", "registration_date"]
)
customers = report_and_drop_duplicates(customers, "customers", subset=["customer_id"])
customers = flag_invalid_range(customers, "customers", "total_orders", min_val=0)
customers = flag_invalid_range(customers, "customers", "avg_order_value", min_val=0)
customers = flag_invalid_range(customers, "customers", "pincode", min_val=100000, max_val=999999)
 
# ---------------------------------------------------------------------
# 2. PRODUCTS
# ---------------------------------------------------------------------
log("\nProcessing blinkit_products.csv ...")
products = pd.read_csv(f"{RAW_DIR}/blinkit_products.csv")
 
products = strip_and_standardize_strings(products, ["product_name", "category", "brand"])
products = report_and_drop_nulls(products, "products", ["product_id", "price"])
products = report_and_drop_duplicates(products, "products", subset=["product_id"])
products = flag_invalid_range(products, "products", "price", min_val=0)
products = flag_invalid_range(products, "products", "mrp", min_val=0)
products = flag_invalid_range(products, "products", "margin_percentage", min_val=0, max_val=100)
products = flag_invalid_range(products, "products", "shelf_life_days", min_val=0)
# price should not exceed mrp
bad_price = (products["price"] > products["mrp"]).sum()
if bad_price > 0:
    log(f"[products] {bad_price} rows had price > mrp — dropped.")
    products = products[products["price"] <= products["mrp"]]
 
# ---------------------------------------------------------------------
# 3. ORDERS
# ---------------------------------------------------------------------
log("\nProcessing blinkit_orders.csv ...")
orders = pd.read_csv(f"{RAW_DIR}/blinkit_orders.csv")
 
orders = strip_and_standardize_strings(
    orders, ["delivery_status", "payment_method"]
)
for date_col in ["order_date", "promised_delivery_time", "actual_delivery_time"]:
    orders[date_col] = pd.to_datetime(orders[date_col], errors="coerce")
 
orders = report_and_drop_nulls(
    orders, "orders", ["order_id", "customer_id", "order_date", "store_id"]
)
orders = report_and_drop_duplicates(orders, "orders", subset=["order_id"])
orders = flag_invalid_range(orders, "orders", "order_total", min_val=0)
 
# actual delivery cannot be before order placed
bad_time = (orders["actual_delivery_time"] < orders["order_date"]).sum()
if bad_time > 0:
    log(f"[orders] {bad_time} rows had actual_delivery_time before order_date — dropped.")
    orders = orders[orders["actual_delivery_time"] >= orders["order_date"]]
 
# derived column: delivery delay in minutes (useful downstream, harmless to add now)
orders["delivery_delay_minutes"] = (
    (orders["actual_delivery_time"] - orders["promised_delivery_time"])
    .dt.total_seconds() / 60
)
 
# referential integrity: every order must map to a known customer
valid_customers = set(customers["customer_id"])
before = len(orders)
orders = orders[orders["customer_id"].isin(valid_customers)]
if len(orders) != before:
    log(f"[orders] Removed {before - len(orders)} rows with customer_id "
        f"not found in customers table.")
 
# ---------------------------------------------------------------------
# 4. ORDER ITEMS
# ---------------------------------------------------------------------
log("\nProcessing blinkit_order_items.csv ...")
order_items = pd.read_csv(f"{RAW_DIR}/blinkit_order_items.csv")
 
order_items = report_and_drop_nulls(
    order_items, "order_items", ["order_id", "product_id", "quantity", "unit_price"]
)
order_items = report_and_drop_duplicates(order_items, "order_items")
order_items = flag_invalid_range(order_items, "order_items", "quantity", min_val=1)
order_items = flag_invalid_range(order_items, "order_items", "unit_price", min_val=0)
 
# referential integrity: order_id and product_id must exist in their parent tables
valid_orders = set(orders["order_id"])
valid_products = set(products["product_id"])
before = len(order_items)
order_items = order_items[
    order_items["order_id"].isin(valid_orders)
    & order_items["product_id"].isin(valid_products)
]
if len(order_items) != before:
    log(f"[order_items] Removed {before - len(order_items)} rows with "
        f"order_id/product_id not found in parent tables.")
 
# ---------------------------------------------------------------------
# 5. CUSTOMER FEEDBACK
# ---------------------------------------------------------------------
log("\nProcessing blinkit_customer_feedback.csv ...")
feedback = pd.read_csv(f"{RAW_DIR}/blinkit_customer_feedback.csv")
 
feedback = strip_and_standardize_strings(
    feedback, ["feedback_text", "feedback_category", "sentiment"]
)
feedback["feedback_date"] = pd.to_datetime(feedback["feedback_date"], errors="coerce")
feedback = report_and_drop_nulls(feedback, "feedback", ["feedback_id", "order_id"])
feedback = report_and_drop_duplicates(feedback, "feedback", subset=["feedback_id"])
feedback = flag_invalid_range(feedback, "feedback", "rating", min_val=1, max_val=5)
 
before = len(feedback)
feedback = feedback[feedback["order_id"].isin(valid_orders)]
if len(feedback) != before:
    log(f"[feedback] Removed {before - len(feedback)} rows with order_id "
        f"not found in orders table.")
 
# ---------------------------------------------------------------------
# 6. MARKETING PERFORMANCE
# ---------------------------------------------------------------------
log("\nProcessing blinkit_marketing_performance.csv ...")
marketing = pd.read_csv(f"{RAW_DIR}/blinkit_marketing_performance.csv")
 
marketing = strip_and_standardize_strings(
    marketing, ["campaign_name", "target_audience", "channel"]
)
marketing["date"] = pd.to_datetime(marketing["date"], errors="coerce")
marketing = report_and_drop_nulls(marketing, "marketing", ["campaign_id", "date"])
marketing = report_and_drop_duplicates(marketing, "marketing")
marketing = flag_invalid_range(marketing, "marketing", "impressions", min_val=0)
marketing = flag_invalid_range(marketing, "marketing", "clicks", min_val=0)
marketing = flag_invalid_range(marketing, "marketing", "conversions", min_val=0)
marketing = flag_invalid_range(marketing, "marketing", "spend", min_val=0)
 
# clicks should not exceed impressions; conversions should not exceed clicks
bad = (marketing["clicks"] > marketing["impressions"]).sum()
if bad > 0:
    log(f"[marketing] {bad} rows had clicks > impressions — dropped.")
    marketing = marketing[marketing["clicks"] <= marketing["impressions"]]
bad = (marketing["conversions"] > marketing["clicks"]).sum()
if bad > 0:
    log(f"[marketing] {bad} rows had conversions > clicks — dropped.")
    marketing = marketing[marketing["conversions"] <= marketing["clicks"]]
 
# ---------------------------------------------------------------------
# SAVE CLEANED FILES
# ---------------------------------------------------------------------
outputs = {
    "blinkit_customers_cleaned.csv": customers,
    "blinkit_products_cleaned.csv": products,
    "blinkit_orders_cleaned.csv": orders,
    "blinkit_order_items_cleaned.csv": order_items,
    "blinkit_customer_feedback_cleaned.csv": feedback,
    "blinkit_marketing_performance_cleaned.csv": marketing,
}
 
log("\n" + "=" * 50)
log("FINAL ROW COUNTS (after cleaning)")
log("=" * 50)
for fname, df in outputs.items():
    df.to_csv(f"{OUT_DIR}/{fname}", index=False)
    log(f"{fname}: {len(df)} rows")
 
with open(f"{OUT_DIR}/cleaning_report.txt", "w") as f:
    f.write("\n".join(report_lines))
 
log(f"\nCleaned files + report saved to: {OUT_DIR}")
 

Processing blinkit_customers.csv ...
[customers] 223 rows had 'pincode' outside [100000, 999999] — dropped.

Processing blinkit_products.csv ...

Processing blinkit_orders.csv ...
[orders] Removed 462 rows with customer_id not found in customers table.

Processing blinkit_order_items.csv ...
[order_items] Removed 462 rows with order_id/product_id not found in parent tables.

Processing blinkit_customer_feedback.csv ...
[feedback] Removed 462 rows with order_id not found in orders table.

Processing blinkit_marketing_performance.csv ...

FINAL ROW COUNTS (after cleaning)
blinkit_customers_cleaned.csv: 2277 rows
blinkit_products_cleaned.csv: 268 rows
blinkit_orders_cleaned.csv: 4538 rows
blinkit_order_items_cleaned.csv: 4538 rows
blinkit_customer_feedback_cleaned.csv: 4538 rows
blinkit_marketing_performance_cleaned.csv: 5400 rows

Cleaned files + report saved to: ./cleaned
